# Clustering
You will be asked to implement several functions.Team work is not allowed. 

Everybody implements his/her own code. Discussing issues with others is fine, sharing code with others is not.

If you use any code fragments found on the Internet, make sure you reference them properly.

In this exercise we will use unsupervised methods for clustering, in this case instead of having $X$ and $Y$ pairs at train time, we only have the input data $X$ at train time.

Let's work with 2 popular methods: K-means and Mean-shift.

## Dataset
3. Iris Dataset
A classical Plant classification dataset. It contains 3 classes (Iris Setosa, Iris Versicolour, Iris Virginica) and 4 parameters (sepal length, sepal width, petal length, petal width all in cm)

## Objectives
1. Apply previos learned methods to a different dataset
2. Understand how to evaluate them by intuition and metric values
3. implement your own mean shift algorithm

## Contents:
 

1) Iris dataset (4 points)

2) Metric based evaluation (4 points)

2) Mean shift implementation (4 points)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.cluster import MeanShift
%matplotlib inline 

## 1 Iris dataset
Discover the iris dataset, apply kmeans and meanshift similar to the last lab

In [ ]:
# Load data
df_iris=pd.read_csv('data/iris.data', header=None, names=['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species'])

## Seperate Test set
This time we have labels to our dataset as well. These can be used to quantify the clustering performance only use the train part to fit your clustering

In [ ]:
# DO NOT CHANGE
iris_test = df_iris.sample(frac = 0.2, random_state=42) 
iris_train = df_iris.drop(iris_test.index)

### a) Discover the dataset (1 point)
Here we check the statistics and ranges of the individual datapoints and also make sure there are no none or outlier.

Plot the distribution (4 plots), datatypes, shape and statistics

### b) Kmeans (1 point)
- Cluster, print and visualize your predictions using 3 features (try at least 2 combinations, visualize with the provided function) 


- Fit a model using all features and print the resulting clusters. This model will be used for Metric evaluation

In [ ]:
def visualise(data, idx):
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(data.values[:,0],data.values[:,1],data.values[:,2],marker='.', c=idx)
    ax.set_xlabel(data.columns[0])
    ax.set_ylabel(data.columns[1])
    ax.set_zlabel(data.columns[2])  
    plt.show()

### b) Meanshift (1 point)

Fit a model using all features and print the resulting clusters. This model will be used for Metric evaluation

### Question (1 point)
Which hyperparameter did you use? why did you choose them?

which method performs better? Explain how you come to this conclusion based on the parts above (without looking at the metrics).

## 2) Metric based Evaluation (4 points)

### Implementatinon (3 points)
First you will implement some useful metrics which can help you measure the performance of your algorithms
Then use remaining 50 samples to figure out how your model performs on unseen data

In [ ]:
# complete these functions
def calculate_majority_label(cluster:pd.Series)-> str: 
    # Calculate to which label the cluster belongs (which class occurs most often)
    return 
def coverage_of_class(cluster:pd.Series, dataset:pd.Series)-> float: 
    # calculate which percentage of the majority label is coverad by this cluster 
    # with respect to the number of occurence in the whole dataset
    return 
def purity(cluster:pd.Series)-> float:
    # calculate the percentage of the majority class within the cluster
    return



In [ ]:
# DO NOT CHANGE
print('Kmeans')
clusters = np.unique(y_test_kmeans)
for i in clusters:
    print('cluster: ', i)
    cluster = iris_test['species'][y_test_kmeans==i]
    print(f'cluster {i} has a majority label of: {calculate_majority_label(iris_train["species"][y_kmeans==i])} in train label,')
    print(f'cluster {i} has a majority label of: {calculate_majority_label(cluster)},')
    print(f'covers {coverage_of_class(cluster, iris_test["species"])*100}% of this class,')
    print(f'has a purity of {purity(cluster)*100}%')

print('MeanShift')
clusters = np.unique(y_test_meanshift)
for i in clusters:
    print('cluster: ', i)
    cluster = iris_test['species'][y_test_meanshift==i]
    print(f'cluster {i} has a majority label of: {calculate_majority_label(iris_train["species"][y_meanshift==i])} in train label,')
    print(f'cluster {i} has a majority label of: {calculate_majority_label(cluster)},')
    print(f'covers {coverage_of_class(cluster, iris_test["species"])*100}% of this class,')
    print(f'has a purity of {purity(cluster)*100}%')

### Question (1 point):
- Did the metric confirm your intuition? 
- Why are metrics useful?
- Can they be missleading as well?
- When does the purity get less reliable? 
- When does the coverage get less reliable?

Hint: Does the size of the clusters effect the metrics?

Explain your answers.

## 3) Your own Mean-shift implementation (4 points)

Let's implement mean-shift algorithm ourselves.

Complete the function ml_meanshift that performs the mean shift algorithm. The function takes 3 arguments:

- input data: sample points in a N-by-2 matrix (number of rows is the number of samples, dimensionality of the input data will always be 2 for this exercise)
- the kernel bandwidth $h$
- the stopping threshold $\vartheta$

It should return two values:

- cluster indexes: a column vector with N rows, specifying the cluster index for each sample
- cluster modes: a M-by-2 matrix, returning the cluster modes (the points with the highest density) for each cluster (where M is the number of clusters)

For this task, use the Epanechnikov kernel. Luckily all terms before the sum cancel out in the mean shift formula, leading to

$$
\mathbf{q}_{t+1} = \frac{
\sum_{i=1}^N \mathbf{x}_i
\max\left(0, 1 - \frac{|\mathbf{q}_t - \mathbf{x}i|^2}{h^2} \right)
} {
\sum_{i=1}^N
\max\left(0, 1 - \frac{|\mathbf{q}_t - \mathbf{x}_i|^2}{h^2} \right)
}
$$

Start the mean shift procedure at each point and iterate until $|\mathbf{q}_{t}-\mathbf{q}_{t-1}| < \vartheta$ where $\vartheta$ is the threshold passed to the function. Additionally also count the iterations and add it as condtion so the loop does not run infinitely. You can use $200$ as the maximum value of iteration.

When the iteration stopped, decide if a cluster mode already exists that is closer than $\frac{h}{5}$. If yes, assign the point that you started at to this cluster. Otherwise, create a new cluster and assign the point to the new cluster.

In [ ]:
import math
from numpy import linalg as la
import numpy.matlib

In [ ]:
# DO NOT MODIFY
def visualise_kmeans(data,idx,centers,updated=None):
    fig = plt.figure()
    ax = fig.add_subplot(1,1,1)
    ax.scatter(data[:,0],data[:,1],marker='.', c=idx)
    ax.plot(centers[:,0],centers[:,1],'+',color='r',markersize=15,mew=2)
    plt.show()
def test_meanshift():
    data = np.genfromtxt('data/toy_data.csv', delimiter=',')
    idx, centers = ml_meanshift(data,4,0.001)
    
    centers_2 = np.array(centers)
    
    visualise_kmeans(data=data,idx=idx,centers=centers_2)

In [ ]:
def ml_meanshift(data,h,theta):
    # ml_meanshift returns cluster indices and modes from computed with meanshift algorithm

    # initialization
    #initialize modes
    modes = 
    #initialize index variable for data points
    index = 
    #s = number of data points
    s = 

    #take i-th element as query point
    for i in range(s):
        #initialize starting q0 and q1 condition
        q1 = 
        q0 = 

        #iteration initilization
        it = 0

        while ():
            #update break criterion values

            #update iteration
            it = it+1

        #check for clusters in the vicinity
        #1st condition: if mode is not empty
        #2nd: if a cluster mode exists closer than h/5
        #else create a new cluster

    return index, modes

In [ ]:
# test and visualise your results
test_meanshift()